# PhysioRisk vs PhenoAge(_noCRP_proxy) — Aligned Benchmark Comparison Notebook

This notebook performs a benchmark comparison between the locked **2-axis PhysioRisk** model and a **Levine-style no-CRP comparator** on a benchmark-aligned NHANES cohort.

## Contract

### Inputs
Frozen baseline-scored train/test parquet files for the aligned benchmark cohort.  
These files must already contain:

- `BaselineRisk`
- `PhysioRisk_2axis`
- `TotalRisk_2axis`
- Levine-noCRP biomarker fields required for comparator construction

### Invariants
- No baseline refitting occurs in this notebook
- No PhysioRisk rebuilding occurs in this notebook
- Comparator preprocessing is fit on **training data only**
- `PhenoAgeAccel` is constructed using **train-frozen spline(age)+sex residualization**

### Outputs
- benchmark comparison table
- optional compact summary
- optional CSV export

In [ ]:
!pip install lifelines

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

from lifelines import CoxPHFitter
from lifelines.utils import concordance_index


## Configuration


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# -------------------------------------------------
# USER CONFIG
# -------------------------------------------------

TRAIN_PATH = Path("/content/drive/MyDrive/LRE/data/pub/physiorisk_2axis_v1_train_scored_from_frozen_baseline_levinenocrp.parquet")
TEST_PATH  = Path("/content/drive/MyDrive/LRE/data/pub/physiorisk_2axis_v1_test_scored_from_frozen_baseline_levinenocrp.parquet")

ID_COL = "SEQN"
TIME_COL = "PERMTH_INT"
EVENT_COL = "MORTSTAT"
AGE_COL = "RIDAGEYR"
SEX_COL = "RIAGENDR"

REQUIRED_SCORE_COLS = [
    "BaselineRisk",
    "PhysioRisk_2axis",
    "TotalRisk_2axis",
]

# Levine / PhenoAge biomarker columns expected in the input cohort
PHENO_COLS = [
    "LBXSAL",    # albumin
    "LBXSCR",    # creatinine
    "LBXGLU",    # glucose
    "LBXMCV",    # mean cell volume
    "LBXRDW",    # red cell distribution width
    "LBXWBCSI",  # white blood cell count
    "LBXLYPCT",  # lymphocyte percent
    "LBXSAPSI",  # alkaline phosphatase
]

print(TRAIN_PATH)
print(TEST_PATH)


/content/drive/MyDrive/LRE/data/pub/physiorisk_2axis_v1_train_scored_from_frozen_baseline_levinenocrp.parquet
/content/drive/MyDrive/LRE/data/pub/physiorisk_2axis_v1_test_scored_from_frozen_baseline_levinenocrp.parquet


## Load aligned benchmark cohort


In [ ]:
train_df = pd.read_parquet(TRAIN_PATH).copy()
test_df  = pd.read_parquet(TEST_PATH).copy()

required_cols = [ID_COL, TIME_COL, EVENT_COL, AGE_COL, SEX_COL] + REQUIRED_SCORE_COLS + PHENO_COLS
missing_train = [c for c in required_cols if c not in train_df.columns]
missing_test  = [c for c in required_cols if c not in test_df.columns]

if missing_train:
    raise ValueError(f"Missing required columns in train_df: {missing_train}")
if missing_test:
    raise ValueError(f"Missing required columns in test_df: {missing_test}")

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("Train deaths:", int(train_df[EVENT_COL].sum()))
print("Test deaths :", int(test_df[EVENT_COL].sum()))


Train shape: (13090, 19)
Test shape : (6739, 19)
Train deaths: 2520
Test deaths : 416


## Levine-style no-CRP comparator

This notebook constructs a **Levine-style no-CRP proxy** by applying the published PhenoAge-style linear predictor with the CRP term removed.

This score is used as a **benchmark comparator** only.  
It should not be interpreted as an exact reconstruction of the published PhenoAge model.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from patsy import dmatrix, build_design_matrices


# Levine-style no-CRP proxy coefficients:
# Original PhenoAge formula with the CRP term removed.
# Keep this clearly labeled as a proxy unless/until you refit a true no-CRP model.
PHENO_NOCRP_COEFS = {
    "LBXSAL":   -0.0336,
    "LBXSCR":    0.0095,
    "LBXGLU":    0.1953,
    "LBXLYPCT": -0.0120,
    "LBXMCV":    0.0268,
    "LBXRDW":    0.3306,
    "LBXSAPSI":  0.0019,
    "LBXWBCSI":  0.0554,
    AGE_COL:     0.0804,
}
PHENO_INTERCEPT = -19.907

# ----------------------------
# 1) Train-derived imputation
# ----------------------------
def impute_pheno_inputs(train_df: pd.DataFrame, test_df: pd.DataFrame, pheno_cols: list[str]):
    imputer = SimpleImputer(strategy="median")
    train_imp = train_df.copy()
    test_imp = test_df.copy()

    train_imp[pheno_cols] = imputer.fit_transform(train_imp[pheno_cols])
    test_imp[pheno_cols] = imputer.transform(test_imp[pheno_cols])

    return train_imp, test_imp, imputer

# ----------------------------
# 2) Build PhenoAge no-CRP proxy
# ----------------------------
def add_phenoage_no_crp_proxy(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Levine-style linear predictor with CRP term removed
    xb = (
        PHENO_INTERCEPT
        + PHENO_NOCRP_COEFS["LBXSAL"]   * out["LBXSAL"]
        + PHENO_NOCRP_COEFS["LBXSCR"]   * out["LBXSCR"]
        + PHENO_NOCRP_COEFS["LBXGLU"]   * np.log(out["LBXGLU"].clip(lower=1e-6))
        + PHENO_NOCRP_COEFS["LBXLYPCT"] * out["LBXLYPCT"]
        + PHENO_NOCRP_COEFS["LBXMCV"]   * out["LBXMCV"]
        + PHENO_NOCRP_COEFS["LBXRDW"]   * out["LBXRDW"]
        + PHENO_NOCRP_COEFS["LBXSAPSI"] * out["LBXSAPSI"]
        + PHENO_NOCRP_COEFS["LBXWBCSI"] * out["LBXWBCSI"]
        + PHENO_NOCRP_COEFS[AGE_COL]    * out[AGE_COL]
    )

    # Keep this as the benchmark score.
    # Name it explicitly so you do not accidentally overclaim it as full PhenoAge.
    out["PhenoAge_noCRP_proxy"] = xb

    return out

# ----------------------------
# 3) Train-frozen residualization
#    PhenoAgeAccel = residual from spline(age)+sex model
# ----------------------------
def add_phenoage_accel_spline_sex(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    pheno_col: str = "PhenoAge_noCRP_proxy",
    age_col: str = AGE_COL,
    sex_col: str = SEX_COL,
):
    formula = f"cr({age_col}, df=4) + C({sex_col})"

    X_train = dmatrix(formula, train_df, return_type="dataframe")
    design_info = X_train.design_info
    X_test = build_design_matrices([design_info], test_df, return_type="dataframe")[0]

    reg = LinearRegression()
    reg.fit(X_train, train_df[pheno_col])

    train_out = train_df.copy()
    test_out = test_df.copy()

    train_out["PhenoAgeAccel"] = train_out[pheno_col] - reg.predict(X_train)
    test_out["PhenoAgeAccel"] = test_out[pheno_col] - reg.predict(X_test)

    return train_out, test_out, reg, design_info

# ----------------------------
# 4) Run the pipeline
# ----------------------------
train_df, test_df, pheno_imputer = impute_pheno_inputs(train_df, test_df, PHENO_COLS)

train_df = add_phenoage_no_crp_proxy(train_df)
test_df = add_phenoage_no_crp_proxy(test_df)

train_df, test_df, pheno_accel_model, pheno_accel_design = add_phenoage_accel_spline_sex(
    train_df,
    test_df,
    pheno_col="PhenoAge_noCRP_proxy",
)

# Optional: if you want a z-scored version for plots/tables
pheno_train_mean = train_df["PhenoAge_noCRP_proxy"].mean()
pheno_train_std = train_df["PhenoAge_noCRP_proxy"].std()

if pheno_train_std > 0 and not pd.isna(pheno_train_std):
    train_df["PhenoAge_noCRP_proxy_z"] = (train_df["PhenoAge_noCRP_proxy"] - pheno_train_mean) / pheno_train_std
    test_df["PhenoAge_noCRP_proxy_z"] = (test_df["PhenoAge_noCRP_proxy"] - pheno_train_mean) / pheno_train_std

# Quick sanity checks
print("Train corr(age, PhenoAgeAccel):", train_df[[AGE_COL, "PhenoAgeAccel"]].corr().iloc[0, 1])
print("Test  corr(age, PhenoAgeAccel):", test_df[[AGE_COL, "PhenoAgeAccel"]].corr().iloc[0, 1])

Train corr(age, PhenoAgeAccel): 1.5942624454329943e-15
Test  corr(age, PhenoAgeAccel): 0.03329887535241356


## PhenoAgeAccel (train-frozen spline age + sex residual)

`PhenoAgeAccel` is defined here as the residual from a **train-frozen linear model of PhenoAge_noCRP_proxy on spline(age) + sex**.

This preserves train/test discipline and creates a comparator that is age-adjusted using the same general age/sex basis used elsewhere in the project.

## Evaluation helpers


In [ ]:
def c_index_for_score(df, score_col, time_col, event_col):
    tmp = df[[time_col, event_col, score_col]].dropna().copy()
    return concordance_index(tmp[time_col], -tmp[score_col], tmp[event_col])

from lifelines import CoxPHFitter

def hr_per_1sd(df, score_col, time_col=TIME_COL, event_col=EVENT_COL):
    tmp = df[[time_col, event_col, score_col]].dropna().copy()

    sd = tmp[score_col].std()
    if sd == 0 or pd.isna(sd):
        return {
            "HR_per_1SD": np.nan,
            "HR_95CI_low": np.nan,
            "HR_95CI_high": np.nan,
            "p": np.nan,
        }

    zcol = f"{score_col}_z"
    tmp[zcol] = (tmp[score_col] - tmp[score_col].mean()) / sd

    cph = CoxPHFitter()
    cph.fit(tmp[[time_col, event_col, zcol]], duration_col=time_col, event_col=event_col)

    row = cph.summary.loc[zcol]
    return {
        "HR_per_1SD": float(row["exp(coef)"]),
        "HR_95CI_low": float(row["exp(coef) lower 95%"]),
        "HR_95CI_high": float(row["exp(coef) upper 95%"]),
        "p": float(row["p"]),
    }

def evaluate_scores(df, cohort_name, score_cols, time_col, event_col, age_col):
    rows = []
    for score_col in score_cols:
        hr = hr_per_1sd(df, score_col, time_col, event_col)

        # Calculate tmp_corr and corr_age outside the dictionary creation
        tmp_corr = df[[score_col, age_col]].dropna()
        corr_age = np.nan if len(tmp_corr) == 0 else float(tmp_corr[score_col].corr(tmp_corr[age_col]))

        rows.append({
            "cohort": cohort_name,
            "risk_score": score_col,
            "C_index": c_index_for_score(df, score_col, time_col, event_col),
            "HR_per_1SD": hr["HR_per_1SD"],
            "HR_95CI_low": hr["HR_95CI_low"],
            "HR_95CI_high": hr["HR_95CI_high"],
            "p": hr["p"],
            "corr_age": corr_age, # Add corr_age as a key-value pair
        })
    return pd.DataFrame(rows)


## Main aligned comparison


In [ ]:
score_cols = [
    "BaselineRisk",
    "PhysioRisk_2axis",
    "TotalRisk_2axis",
    "PhenoAge_noCRP_proxy",
    "PhenoAgeAccel",
]

eval_train = evaluate_scores(
    df=train_df,
    cohort_name="train_benchmark",
    score_cols=score_cols,
    time_col=TIME_COL,
    event_col=EVENT_COL,
    age_col=AGE_COL,
)

eval_test = evaluate_scores(
    df=test_df,
    cohort_name="test_benchmark",
    score_cols=score_cols,
    time_col=TIME_COL,
    event_col=EVENT_COL,
    age_col=AGE_COL,
)

benchmark_table = pd.concat([eval_test, eval_train], ignore_index=True)
benchmark_table


,cohort,risk_score,C_index,HR_per_1SD,HR_95CI_low,HR_95CI_high,p,corr_age
0,test_benchmark,BaselineRisk,0.826876,3.735672,3.346768,4.169769,4.376900e-122,9.697340e-01
1,test_benchmark,PhysioRisk_2axis,0.654110,2.771658,2.655681,2.892700,0.000000e+00,1.062335e-01
2,test_benchmark,TotalRisk_2axis,0.839760,1.877467,1.815235,1.941832,1.077953e-293,8.109274e-01
3,test_benchmark,PhenoAge_noCRP_proxy,0.841108,3.976118,3.636924,4.346946,3.398262e-202,9.613695e-01
4,test_benchmark,PhenoAgeAccel,0.637857,1.432148,1.359932,1.508198,3.701780e-42,3.329888e-02
5,train_benchmark,BaselineRisk,0.854595,4.286222,4.097474,4.483664,0.000000e+00,9.714474e-01
6,train_benchmark,PhysioRisk_2axis,0.602373,1.333612,1.294880,1.373502,1.068879e-81,-1.432347e-16
7,train_benchmark,TotalRisk_2axis,0.857525,2.426656,2.371736,2.482849,0.000000e+00,7.903362e-01
8,train_benchmark,PhenoAge_noCRP_proxy,0.864335,3.669860,3.555910,3.787461,0.000000e+00,9.577213e-01
9,train_benchmark,PhenoAgeAccel,0.585610,1.246198,1.219091,1.273907,1.127849e-85,2.044577e-15


## Optional compact side-by-side summary


In [ ]:
summary_cols = ["cohort", "risk_score", "C_index", "HR_per_1SD", "p", "corr_age"]
benchmark_table[summary_cols]


,cohort,risk_score,C_index,HR_per_1SD,p,corr_age
0,test_benchmark,BaselineRisk,0.826876,3.735672,4.376900e-122,9.697340e-01
1,test_benchmark,PhysioRisk_2axis,0.654110,2.771658,0.000000e+00,1.062335e-01
2,test_benchmark,TotalRisk_2axis,0.839760,1.877467,1.077953e-293,8.109274e-01
3,test_benchmark,PhenoAge_noCRP_proxy,0.841108,3.976118,3.398262e-202,9.613695e-01
4,test_benchmark,PhenoAgeAccel,0.637857,1.432148,3.701780e-42,3.329888e-02
5,train_benchmark,BaselineRisk,0.854595,4.286222,0.000000e+00,9.714474e-01
6,train_benchmark,PhysioRisk_2axis,0.602373,1.333612,1.068879e-81,-1.432347e-16
7,train_benchmark,TotalRisk_2axis,0.857525,2.426656,0.000000e+00,7.903362e-01
8,train_benchmark,PhenoAge_noCRP_proxy,0.864335,3.669860,0.000000e+00,9.577213e-01
9,train_benchmark,PhenoAgeAccel,0.585610,1.246198,1.127849e-85,2.044577e-15


## Optional export


In [ ]:
EXPORT_DIR = Path("/content/drive/MyDrive/LRE/data/pub/benchmark_levine")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

TABLE_OUT = EXPORT_DIR / "physiorisk_vs_phenoage_aligned_comparison.csv"

benchmark_table.to_csv(TABLE_OUT, index=False)
print("Exported:", TABLE_OUT)


Exported: /content/drive/MyDrive/LRE/data/pub/benchmark_levine/physiorisk_vs_phenoage_aligned_comparison.csv


In [ ]:
test_compare_export = test_df[
    ["SEQN", "PhenoAge_noCRP_proxy", "PhenoAgeAccel"]
].copy()

test_compare_export.to_parquet(
    "/content/drive/MyDrive/LRE/data/pub/benchmark_levine/test_phenoage_compare_outputs.parquet",
    index=False
)

In [ ]:
print("\nAge correlations:")
print("Train PhenoAgeAccel:", train_df[[AGE_COL, "PhenoAgeAccel"]].corr().iloc[0, 1])
print("Test  PhenoAgeAccel:", test_df[[AGE_COL, "PhenoAgeAccel"]].corr().iloc[0, 1])

print("\nBenchmark scores included:")
print(score_cols)


Age correlations:
Train PhenoAgeAccel: 1.5942624454329943e-15
Test  PhenoAgeAccel: 0.03329887535241356

Benchmark scores included:
['BaselineRisk', 'PhysioRisk_2axis', 'TotalRisk_2axis', 'PhenoAge_noCRP_proxy', 'PhenoAgeAccel']


## Notes

* **PhenoAge implementation**

  * The `PhenoAge_noCRP_proxy` score is constructed using a Levine-style linear predictor with the CRP term removed.
  * This is a **proxy comparator**, not a full refit of the original PhenoAge model.
  * As such, it should be interpreted as a **benchmark-aligned approximation**, not a direct reproduction of published PhenoAge.

* **PhenoAgeAccel definition**

  * `PhenoAgeAccel` is defined as the residual from a **train-frozen linear model of `PhenoAge_noCRP_proxy` on spline-transformed age and sex**:

  * Model: `PhenoAge_noCRP_proxy ~ cr(age, df=4) + sex`
  * Fit: **training data only**
  * Application: the fitted model is applied unchanged to the test set

  Acceleration is computed as:

  > `PhenoAgeAccel = observed PhenoAge_noCRP_proxy − predicted value from the age/sex model`

  This construction ensures that:

  * the resulting score is **orthogonal to age (and sex) in the training data**
  * the same age/sex adjustment is applied consistently to the test set (no leakage)
  * the adjustment uses a **flexible nonlinear age basis**, avoiding underfitting of age effects
  * the procedure is **aligned with the residualization framework used in PhysioRisk**
  * Residualization is not re-fit on the test set.

  As a result, `PhenoAgeAccel` represents **physiological variation beyond age and sex**, directly comparable to the residualized PhysioRisk component.


* **Train/test discipline**

  * All preprocessing steps for comparator construction (imputation and residualization) are fit on the **training set only**.
  * No information from the test set is used during model fitting or transformation.

* **Comparability of scores**

  * `BaselineRisk` and `PhenoAge_noCRP_proxy` are **age-entangled mortality predictors**.
  * `PhysioRisk_2axis` and `PhenoAgeAccel` are **age-adjusted (residual) scores** intended to capture physiological variation beyond age.
  * Direct comparison between these two groups should account for this structural difference.

* **Interpretation of HR per 1 SD**

  * Hazard ratios are computed per **one standard deviation increase**, with standardization performed within each cohort.
  * This allows effect sizes to be compared across scores with different native scales.

* **Cohort alignment**

  * All models are evaluated on the same benchmark-aligned cohort with identical temporal splits.
  * This ensures that differences reflect model behavior rather than cohort differences.

* **Limitations**

  * The no-CRP PhenoAge proxy may not fully preserve the statistical properties of the original model.
  * Residualization-based acceleration metrics can be sensitive to the chosen age model and cohort distribution.
  * Results should be interpreted as **relative comparisons within this aligned benchmark**, not absolute performance estimates.

* **Scope**

  * This notebook is intended for **evaluation and comparison only**.
  * It does not modify or refit the PhysioRisk model or BaselineRisk.
  * Outputs are used for tables and figures in downstream analysis.
